In [ ]:
!pip install -q sentence-transformers faiss-cpu scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.4 MB/s eta 0:00:00


In [2]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 69.0 MB/s eta 0:00:00


In [3]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import time

print("All libraries imported successfully!")

All libraries imported successfully!


In [5]:
documents = [
    # Technology (10)
    "Python is a popular programming language for data analysis.",
    "Machine learning helps computers learn patterns from data.",
    "Artificial intelligence enables machines to perform intelligent tasks.",
    "Cloud computing provides computing resources over the internet.",
    "Cybersecurity protects systems from digital threats.",
    "Data science combines statistics, programming, and domain knowledge.",
    "Neural networks are inspired by the structure of the human brain.",
    "Deep learning uses multiple layers of neural networks.",
    "Natural language processing helps computers understand human language.",
    "Computer vision allows machines to analyze images and videos.",

    # Sports (10)
    "Football players train regularly to improve their performance.",
    "Cricket teams practice batting and bowling before tournaments.",
    "Basketball requires good coordination and teamwork.",
    "Tennis players need speed and accuracy during matches.",
    "Athletes follow training schedules before competitions.",
    "Swimming is a sport that requires strength and endurance.",
    "The goalkeeper stopped the ball during the football match.",
    "The cricket player scored a century in the final match.",
    "The basketball team won the championship.",
    "Running regularly can improve athletic performance.",

    # Health (10)
    "Regular exercise helps maintain physical fitness.",
    "Drinking enough water is important for the body.",
    "A balanced diet provides essential nutrients.",
    "Sleep is important for physical and mental recovery.",
    "Walking every day can support a healthy lifestyle.",
    "Fruits and vegetables provide important vitamins.",
    "Doctors recommend regular health checkups.",
    "Meditation can help improve relaxation and focus.",
    "Healthy food choices can support overall wellness.",
    "Physical activity can improve cardiovascular health.",

    # Travel (10)
    "Travelers often book hotels before visiting a new city.",
    "Tourists enjoy exploring historical monuments.",
    "Airplanes make long distance travel faster.",
    "Mountains attract visitors because of their natural beauty.",
    "Travel guides provide useful information about destinations.",
    "Many tourists enjoy visiting beaches during holidays.",
    "Train journeys can be an enjoyable way to explore places.",
    "People often take photographs while traveling.",
    "International travel requires proper documentation.",
    "Tourists usually plan their itinerary before a trip.",

    # Education (10)
    "Students attend classes to learn new concepts.",
    "Online courses help people learn from home.",
    "Teachers explain difficult topics to their students.",
    "Reading books can improve knowledge and vocabulary.",
    "Exams are used to evaluate student learning.",
    "Libraries provide access to many educational resources.",
    "Students often use computers for research and assignments.",
    "Group projects help students develop teamwork skills.",
    "Practical exercises help learners understand concepts.",
    "Education plays an important role in personal development."
]

print("Total documents:", len(documents))

Total documents: 50


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

start_time = time.time()

embeddings = model.encode(documents)

# NumPy float32 array
embeddings = np.array(embeddings, dtype="float32")

end_time = time.time()

print("Embedding shape:", embeddings.shape)
print("Data type:", embeddings.dtype)
print("Embedding time:", round(end_time - start_time, 4), "seconds")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (50, 384)
Data type: float32
Embedding time: 0.4014 seconds


In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("FAISS index created successfully!")
print("Index size:", index.ntotal)

FAISS index created successfully!
Index size: 50


In [ ]:
def semantic_search(query, top_k=3):

    # Convert query into embedding
    query_embedding = model.encode([query])
    query_embedding = np.array(query_embedding, dtype="float32")

    # Search nearest documents in FAISS
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append((documents[idx], distance))

    return results


# Test semantic search
query = "How can computers learn from information?"

results = semantic_search(query, top_k=3)

print("Query:", query)
print("\nTop 3 semantic search results:\n")

for document, distance in results:
    print(f"Distance: {distance:.4f} → {document}")

Query: How can computers learn from information?

Top 3 semantic search results:

Distance: 0.8019 → Machine learning helps computers learn patterns from data.
Distance: 1.0117 → Natural language processing helps computers understand human language.
Distance: 1.0787 → Students often use computers for research and assignments.


In [9]:
def semantic_search(query, top_k=3):

    # Query converting into embedding
    query_embedding = model.encode([query])

    # float32 for FAISS
    query_embedding = np.array(
        query_embedding,
        dtype="float32"
    )

    # search nearest documents in FAISS
    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        results.append(
            (documents[idx], float(distance))
        )

    return results

In [10]:
query = "How can computers learn patterns?"

results = semantic_search(query, top_k=3)

print("Query:", query)
print("\nTop 3 Semantic Search Results:\n")

for document, distance in results:
    print(f"Distance: {distance:.4f} → {document}")

Query: How can computers learn patterns?

Top 3 Semantic Search Results:

Distance: 0.6511 → Machine learning helps computers learn patterns from data.
Distance: 1.0146 → Natural language processing helps computers understand human language.
Distance: 1.1421 → Computer vision allows machines to analyze images and videos.


In [11]:
tfidf_vectorizer = TfidfVectorizer()

tfidf_matrix = tfidf_vectorizer.fit_transform(documents)


def keyword_search(query, top_k=3):

    # Query को TF-IDF vector में convert करना
    query_vector = tfidf_vectorizer.transform([query])

    # Documents के साथ similarity निकालना
    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    )[0]

    # सबसे ज्यादा score वाले documents
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in top_indices:
        results.append(
            (documents[idx], float(scores[idx]))
        )

    return results

In [12]:
query = "machine learning"

results = keyword_search(query, top_k=3)

print("Query:", query)
print("\nTop 3 Keyword Search Results:\n")

for document, score in results:
    print(f"Score: {score:.4f} → {document}")

Query: machine learning

Top 3 Keyword Search Results:

Score: 0.5237 → Machine learning helps computers learn patterns from data.
Score: 0.2203 → Exams are used to evaluate student learning.
Score: 0.2023 → Deep learning uses multiple layers of neural networks.


In [13]:
queries = [
    "How do computers learn patterns?",
    "How can I protect my computer from online attacks?",
    "What helps people recover physically and mentally?",
    "What is a good way to study from home?",
    "I want to visit beautiful natural places.",
    "How can I improve my athletic performance?",
    "How can machines understand human language?",
    "What helps maintain a healthy body?",
    "How do people plan trips to new places?",
    "How can students gain practical knowledge?"
]

print("Total test queries:", len(queries))

Total test queries: 10


In [14]:
comparison_results = []

for query in queries:

    # Semantic Search
    semantic_result = semantic_search(query, top_k=1)[0]

    # Keyword Search
    keyword_result = keyword_search(query, top_k=1)[0]

    comparison_results.append({
        "Query": query,
        "Semantic Result": semantic_result[0],
        "Semantic Distance": round(semantic_result[1], 4),
        "Keyword Result": keyword_result[0],
        "Keyword Score": round(keyword_result[1], 4)
    })


comparison_df = pd.DataFrame(comparison_results)

comparison_df

,Query,Semantic Result,Semantic Distance,Keyword Result,Keyword Score
0,How do computers learn patterns?,Machine learning helps computers learn pattern...,0.6609,Machine learning helps computers learn pattern...,0.6222
1,How can I protect my computer from online atta...,Cybersecurity protects systems from digital th...,1.0281,Online courses help people learn from home.,0.3964
2,What helps people recover physically and menta...,Sleep is important for physical and mental rec...,0.8833,People often take photographs while traveling.,0.2559
3,What is a good way to study from home?,Online courses help people learn from home.,0.6937,Online courses help people learn from home.,0.3236
4,I want to visit beautiful natural places.,Mountains attract visitors because of their na...,0.8595,Train journeys can be an enjoyable way to expl...,0.3249
5,How can I improve my athletic performance?,Running regularly can improve athletic perform...,0.6089,Running regularly can improve athletic perform...,0.7797
6,How can machines understand human language?,Natural language processing helps computers un...,0.7788,Natural language processing helps computers un...,0.5947
7,What helps maintain a healthy body?,Regular exercise helps maintain physical fitness.,0.7975,Regular exercise helps maintain physical fitness.,0.3968
8,How do people plan trips to new places?,Tourists usually plan their itinerary before a...,0.6718,Students attend classes to learn new concepts.,0.2647
9,How can students gain practical knowledge?,Practical exercises help learners understand c...,0.6213,Reading books can improve knowledge and vocabu...,0.3296


In [15]:
mismatch_queries = [
    "How can machines understand human language?",
    "What helps people recover physically and mentally?",
    "How can I protect my computer from online attacks?"
]

for query in mismatch_queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)

    print("\nSemantic Search:")
    for document, distance in semantic_search(query, top_k=1):
        print(f"Distance: {distance:.4f} → {document}")

    print("\nKeyword Search:")
    for document, score in keyword_search(query, top_k=1):
        print(f"Score: {score:.4f} → {document}")


QUERY: How can machines understand human language?

Semantic Search:
Distance: 0.7788 → Natural language processing helps computers understand human language.

Keyword Search:
Score: 0.5947 → Natural language processing helps computers understand human language.

QUERY: What helps people recover physically and mentally?

Semantic Search:
Distance: 0.8833 → Sleep is important for physical and mental recovery.

Keyword Search:
Score: 0.2559 → People often take photographs while traveling.

QUERY: How can I protect my computer from online attacks?

Semantic Search:
Distance: 1.0281 → Cybersecurity protects systems from digital threats.

Keyword Search:
Score: 0.3964 → Online courses help people learn from home.


In [16]:
keyword_queries = [
    "machine learning",
    "cricket player",
    "Python data analysis"
]

for query in keyword_queries:

    print("\n" + "=" * 70)
    print("QUERY:", query)

    print("\nSemantic Search:")
    for document, distance in semantic_search(query, top_k=1):
        print(f"Distance: {distance:.4f} → {document}")

    print("\nKeyword Search:")
    for document, score in keyword_search(query, top_k=1):
        print(f"Score: {score:.4f} → {document}")


QUERY: machine learning

Semantic Search:
Distance: 0.7049 → Machine learning helps computers learn patterns from data.

Keyword Search:
Score: 0.5237 → Machine learning helps computers learn patterns from data.

QUERY: cricket player

Semantic Search:
Distance: 0.7997 → The cricket player scored a century in the final match.

Keyword Search:
Score: 0.4641 → The cricket player scored a century in the final match.

QUERY: Python data analysis

Semantic Search:
Distance: 0.5704 → Python is a popular programming language for data analysis.

Keyword Search:
Score: 0.6413 → Python is a popular programming language for data analysis.


## Semantic Search vs Keyword Search — Trade-off Analysis

### Semantic Search
Semantic search uses embeddings to understand the meaning of a query. It can find relevant documents even when the query uses different words.

### Keyword Search
Keyword search mainly depends on matching words between the query and documents. It works well when exact keywords are important.

### Semantic Search is useful for:
- Understanding user intent
- Finding semantically similar information
- Recommendations
- Question answering
- Queries with different vocabulary

### Keyword Search is useful for:
- Exact word matching
- Names and specific terms
- IDs and codes
- Simple and fast text search

### Production Trade-offs
Semantic search requires generating embeddings and maintaining a vector index, which adds computational resources and processing time.

Keyword search is simpler and can be efficient for exact matching, but it may miss relevant documents when different words are used.

A production search system can combine semantic and keyword search to improve retrieval quality.